<a href="https://colab.research.google.com/github/AncaraniJuanDiego/ProgramacionFuncional/blob/main/Tp_Programacionfuncional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Trabajo final Programacion Funcional**

Participantes: Agustin Scellato- Andres Morenico - Juan Diego Ancarani - Paul Chavez

# **¿Que buscamos hacer con este trabajo? **

Nuestra idea es entrenar un modelo inteligente, utilizando herramientas matematicas conociendo datos del pasado, para hacer predicciones futuras. En este caso elegimos una tematica interesente y de nuestro gusto, como lo es el futbol.



**1) Importacion de librerias**

In [ ]:
# Importaciones base (ciencia de datos)
import pandas as pd
import numpy as np

# Programación funcional / estructuras para pipelines
from collections import deque, defaultdict

# Modelado
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score


# Gráficos
import matplotlib.pyplot as plt

Adquirir datos historicos

Dado que nuestro dataframe contiene datos historicos, solo es posible predecir aspectos que ya ocurrieron..

In [ ]:
df= pd.read_excel("PremierLeague23-25.xlsx")
df.head()


**2) Conteo de partidos x temporada y representacion de resultados**

In [ ]:
# Función pura: misma entrada -> misma salida, sin efectos secundarios.
def season_from_date(d):
    y = d.year
    # La Premier suele iniciar en agosto (>=8). Si es antes, pertenece a la temporada anterior.
    if d.month >= 8:
        return f"{y}-{str(y+1)[-2:]}"
    else:
        return f"{y-1}-{str(y)[-2:]}"

df["Season"] = df["Date"].map(season_from_date)  # map = aplicar función a una serie
df["Season"].value_counts()

#devuelve--> cantidad de partidos por temporada

In [ ]:
# Contamos resultados y graficamos
counts = df["FinalResult"].value_counts().reindex(["H","D","A"])

plt.figure()
counts.plot(kind="bar")
plt.title("Distribución de resultados (H/D/A)")
plt.xlabel("Resultado")
plt.ylabel("Cantidad de partidos")
plt.show()


**3) Predecir la diferencia de goles en un partido de fútbol usando estadísticas del partido.**

**Caracteristicas:**

MSE (error medio cuadrado) : Mide que tan lejos esta la prediccion de los valores reales.

R2 Ajustado: Explica la variacion % entre las variables (1.0 perfecta)--> (1- SSresiduos / SStotal)

In [ ]:
# Encode 'FinalResult' to numerical values

df['FinalResult_encoded'] = df['FinalResult'].map({'H': 1, 'D': 0, 'A': -1})

#Con estas features vamos a predecir la diferencia de goles.
features=['HomeShots',	'AwayShots',	'HomeFauls',	'AwayFauls',	'HomeCorners',
	'AwayCorners', 'FinalResult_encoded']

#1) resultado numerico--> diferencia de goles (Localgoals-AwayGoals).
X= df[features]
y= df["LocalGoals"]-df["AwayGoals"]

#calcular modelos de entrenamiento (train) y de evaluadoras (test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=42)
sclater = StandardScaler()
X_train_scaled = sclater.fit_transform(X_train)
X_test_scaled = sclater.transform(X_test)

#modelo Mlp regressor (maximo de 500 iteraciones).
mlp= MLPRegressor( hidden_layer_sizes= (50,50), activation= 'relu', solver= 'adam', max_iter= 500, random_state= 42)
mlp.fit(X_train_scaled, y_train)

#evaluar prediccion.
y_pred = mlp.predict(X_test_scaled)
mse= mean_squared_error(y_test, y_pred)
r2= r2_score(y_test, y_pred)
print("MSE: ", mse)
print("R2: ", r2)

**3.1) Este grafico explica el error o Mse va bajando a lo largo del entrenamiento en 160 iteraciones..**


Iteracion--> Cada punto del eje horizontal es una pasada de entrenamiento (una actualización de pesos en la red neuronal).

Loss--> MSE (Error medio cuadratico)

In [ ]:
import matplotlib.pyplot as plt

# losses -> lista o array con la pérdida en cada iteración

plt.figure(figsize=(7,5))

plt.plot(mlp.loss_curve_ , linewidth=2.2, alpha=0.9, label="Loss en cada iteración")

plt.xlabel("Iteración", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("Evolución de la pérdida durante el entrenamiento", fontsize=14)

plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

#A las 20 iteraciones baja coinsiderablemente.

**Interpretacion**: el loss  baja repentinamente a partir de las 20 iteraciones, coinsiderando que el modelo aprende a partir de dichas iteraciones como maximo. Luego se mantiene cierta constancia hasta llegar a su punto minimo.

**4) Grafico de valores reales vs valores predichos**

-si, los puntos predecidos estan simetiricamente alineados de los reales, el modelo predice perfectamente.

-En este caso, hay cierta cercania pero no del todo. Esto explica la intensidad moderada (R:0.73)

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(y_test, y_test, marker= "o", label ="Valor real",s=80, alpha= 0.9)
plt.scatter(y_test, y_pred, marker= "x", label= "Prediccion",s=80, alpha= 0.9)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("valor real")
plt.ylabel("Prediccion")
plt.title("comparacion entre valores reales y predichos")
plt.legend()
plt.grid(True)
plt.show()


**Interpretacion**: Los valores reales y predichos estan cerca entre si, explicando MLP Y R2 que son moderado-alto (Mlp= 1.12 , r2 = 0.73)

**5) grafico (HeatMap) CORRELACION ENTRE VARIABLES**

 vamos a observar la correlacion entre las variables, de esta manera podemos entender que variables podrian predecir correctamente a la otra entendiendo que tal alta es la intensidad

In [ ]:
import seaborn as sb

plt.figure(figsize=(7,5))
sb.heatmap(df[features].corr(), annot = True)
plt.title('Correlacion entre variables entrenadas y predichas')
plt.xlabel('Variables x')
plt.ylabel('Variables y')
plt.legend()
plt.show()

Concluimos que: las variables mejor correlacionados, son:

-HomeShoots- HomeCorners (R= 0.54)--> Posiblemente se debe a que, cuando se lanza un tiro de esquina simepre hay posibilidad de un tiro a puerta.

-AwayShoots- AwayCorners (R=0.57) --> Lo mismo, solo que los partidos del dataSet implicam mas dominancia en este sentido, por lo equipos visitantes.

¿Que indican las correlaciones negativas?--> que la ocurrencia de un evento, implica que la otra no suceda o con menos frecuencia.. Ej:

AwayGoals-HomeShoots(R =-0.16) --> Indica que la conversion de goles del equipo visitante, afecta negativamente los tiros a puerta del equipo local.

**6) Feature Engineering pre-partido con un pipeline funcional** (paul)

Idea
Para evitar “trampa” (usar goles/tarjetas del mismo partido), construimos features solo con el historial previo.

Usamos:

map / filter (funciones de orden superior)
state por equipo (estadísticas acumuladas)
ventanas móviles con deque (últimos N partidos)
funciones “tipo funcional”: cada actualización devuelve un nuevo estado (sin mutar el anterior)

In [ ]:
# Este bloque define **constantes** para el feature engineering “pre-partido”.
#
# - `ROLL_N`: cuántos partidos recientes se usan para promedios móviles (rolling).
# - `STAT_KEYS`: qué estadísticas vamos a guardar por equipo (goles, tiros, faltas, etc.).
# - **Para qué**: centralizar en un solo lugar qué se considera “historial” y hacerlo fácil de modificar.

# Variables y ventana de rolling
STAT_KEYS = [
    "gf","ga",
    "shots_for","shots_against",
    "fouls_for","fouls_against",
    "corners_for","corners_against",
    "yellows_for","yellows_against",
    "reds_for","reds_against",
    "points"
]
ROLL_N = 5

def empty_team_state():
    """Crea un estado inicial para un equipo (sin partidos previos)."""
    return {k:0.0 for k in STAT_KEYS} | {
        "played": 0,
        "roll": {k: deque(maxlen=ROLL_N) for k in STAT_KEYS}
    }

def mean_roll(team_state, k):
    """Promedio de los últimos N valores (si no hay, devuelve 0)."""
    r = team_state["roll"][k]
    return float(np.mean(r)) if len(r) else 0.0

def points_from_result(res, is_home=True):
    """Función pura: convierte H/D/A en puntos para local/visitante."""
    if res == "D":
        return 1
    if (res == "H" and is_home) or (res == "A" and (not is_home)):
        return 3
    return 0

def make_match_features(home_state, away_state):
    """Extrae features pre-partido SOLO con historial previo."""
    played_h = max(home_state["played"], 1)
    played_a = max(away_state["played"], 1)
    feats = {}

    # Promedios por partido (histórico)
    feats["home_ppg"]  = home_state["points"]/played_h
    feats["away_ppg"]  = away_state["points"]/played_a
    feats["home_gfpg"] = home_state["gf"]/played_h
    feats["home_gapg"] = home_state["ga"]/played_h
    feats["away_gfpg"] = away_state["gf"]/played_a
    feats["away_gapg"] = away_state["ga"]/played_a

    # Rolling (últimos N)
    feats["home_gf_roll"]      = mean_roll(home_state,"gf")
    feats["home_ga_roll"]      = mean_roll(home_state,"ga")
    feats["away_gf_roll"]      = mean_roll(away_state,"gf")
    feats["away_ga_roll"]      = mean_roll(away_state,"ga")
    feats["home_shots_roll"]   = mean_roll(home_state,"shots_for")
    feats["away_shots_roll"]   = mean_roll(away_state,"shots_for")
    feats["home_corners_roll"] = mean_roll(home_state,"corners_for")
    feats["away_corners_roll"] = mean_roll(away_state,"corners_for")
    feats["home_yellows_roll"] = mean_roll(home_state,"yellows_for")
    feats["away_yellows_roll"] = mean_roll(away_state,"yellows_for")
    feats["home_reds_roll"]    = mean_roll(home_state,"reds_for")
    feats["away_reds_roll"]    = mean_roll(away_state,"reds_for")

    # Diferencias (home - away)
    feats["ppg_diff"]          = feats["home_ppg"] - feats["away_ppg"]
    feats["gfpg_diff"]         = feats["home_gfpg"] - feats["away_gfpg"]
    feats["gapg_diff"]         = feats["home_gapg"] - feats["away_gapg"]
    feats["shots_roll_diff"]   = feats["home_shots_roll"] - feats["away_shots_roll"]
    feats["corners_roll_diff"] = feats["home_corners_roll"] - feats["away_corners_roll"]
    return feats

def update_team_state(team_state, gf, ga, shots_for, shots_against, fouls_for, fouls_against,
                      corners_for, corners_against, yellows_for, yellows_against, reds_for, reds_against, points):
    """Devuelve un NUEVO estado (estilo funcional)."""
    # 1) copiamos totales y estructuras (para no mutar)
    new_state = {k: team_state[k] for k in STAT_KEYS}
    new_state["played"] = team_state["played"] + 1
    new_state["roll"] = {k: deque(team_state["roll"][k], maxlen=ROLL_N) for k in STAT_KEYS}

    # 2) actualizamos totales
    new_state["gf"] += gf; new_state["ga"] += ga
    new_state["shots_for"] += shots_for; new_state["shots_against"] += shots_against
    new_state["fouls_for"] += fouls_for; new_state["fouls_against"] += fouls_against
    new_state["corners_for"] += corners_for; new_state["corners_against"] += corners_against
    new_state["yellows_for"] += yellows_for; new_state["yellows_against"] += yellows_against
    new_state["reds_for"] += reds_for; new_state["reds_against"] += reds_against
    new_state["points"] += points

    # 3) actualizamos rolling windows
    def push(k, v):
        new_state["roll"][k].append(float(v))

    push("gf", gf); push("ga", ga)
    push("shots_for", shots_for); push("shots_against", shots_against)
    push("fouls_for", fouls_for); push("fouls_against", fouls_against)
    push("corners_for", corners_for); push("corners_against", corners_against)
    push("yellows_for", yellows_for); push("yellows_against", yellows_against)
    push("reds_for", reds_for); push("reds_against", reds_against)
    push("points", points)

    return new_state

**6.1) Orden cronologico esencial para que el historial tenga sentido** (paul)

In [ ]:
# Este bloque recorre los partidos en **orden cronológico** para generar un dataset “pre-partido”.
#
# - **Paso 1**: ordenar por fecha (para simular el tiempo real).
# - **Paso 2**: para cada partido:
#   1) crear features desde el historial (antes del partido)
#   2) guardar la fila con su etiqueta/resultado
#   3) recién ahí actualizar estados con lo ocurrido en ese partido
# - **Por qué es importante**: evita *data leakage* (no usar información futura para entrenar).


df_sorted = df.sort_values("Date").reset_index(drop=True)

# Diccionario: equipo -> estado
team_states = defaultdict(empty_team_state)

rows = []
for r in df_sorted.itertuples(index=False):
    h, a = r.HomeTeam, r.AwayTeam

    # 1) features pre-match (solo pasado)
    feats = make_match_features(team_states[h], team_states[a])

    rows.append({
        "Date": r.Date, "Season": r.Season,
        "HomeTeam": h, "AwayTeam": a,
        **feats,
        "y": r.FinalResult  # target: H / D / A
    })

    # 2) actualizamos estados DESPUÉS del partido
    hp = points_from_result(r.FinalResult, True)
    ap = points_from_result(r.FinalResult, False)

    team_states[h] = update_team_state(
        team_states[h],
        r.LocalGoals, r.AwayGoals,
        r.HomeShots, r.AwayShots,
        r.HomeFauls, r.AwayFauls,
        r.HomeCorners, r.AwayCorners,
        r.HomeYellows, r.AwayYellows,
        r.HomeReds, r.AwayReds,
        hp
    )
    team_states[a] = update_team_state(
        team_states[a],
        r.AwayGoals, r.LocalGoals,
        r.AwayShots, r.HomeShots,
        r.AwayFauls, r.HomeFauls,
        r.AwayCorners, r.HomeCorners,
        r.AwayYellows, r.HomeYellows,
        r.AwayReds, r.HomeReds,
        ap
    )

feat_df = pd.DataFrame(rows)
feat_df.head()

**6.2) Seleccion de columnas numericas** (paul)

In [ ]:
 #**Matriz de confusión**: muestra cuántos H/D/A reales se predijeron como cada clase.
# - **Para qué sirve**: no solo ver “cuánto” acierta, sino “en qué” falla (por ejemplo, confundir empates).

X_cols = [c for c in feat_df.columns if c not in ["Date","Season","HomeTeam","AwayTeam","y"]]

train = feat_df[feat_df.Season == "2023-24"]
test  = feat_df[feat_df.Season == "2024-25"]

X_train, y_train = train[X_cols], train["y"]
X_test,  y_test  = test[X_cols],  test["y"]

# Pipeline (estilo declarativo): escalar + modelo
model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, multi_class="multinomial"))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)
f1m = f1_score(y_test, pred, average="macro")

print("Accuracy:", acc)
print("F1 macro:", f1m)
print("\nReporte:")
print(classification_report(y_test, pred))

**6.3) Matriz de confusion (valores reales vs predichos) (paul)**

In [ ]:
labels = ["H","D","A"]
cm = confusion_matrix(y_test, pred, labels=labels)

plt.figure()
plt.imshow(cm, interpolation="nearest")
plt.title("Matriz de confusión (test 2024–25)")
plt.xticks(range(3), labels)
plt.yticks(range(3), labels)
for i in range(3):
    for j in range(3):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.show()

**7) Prediccion personalizada --> predecir tarjetas amarillas, conociendo datos de un partido.**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

features = ["LocalGoals","AwayGoals","HomeShots","AwayShots","HomeFauls","AwayFauls",
            "HomeCorners","AwayCorners","HomeReds","AwayReds"]

X= df[features]
y= df["HomeYellows"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state= 42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_sclaed = scaler.transform(X_test)

yellow_model = MLPRegressor(hidden_layer_sizes= (50,50), max_iter = 500, random_state = 42)
yellow_model.fit(X_train_scaled, y_train)


In [ ]:
def predecir_tarjetas(HomeTeam, AwayTeam, LocalGoals, AwayGoals, Final_result,
                      HomeShots, AwayShots, HomeFauls, AwayFauls,
                      HomeCorners, AwayCorners, HomeReds, AwayReds):

    # Crear el DataFrame
    data = pd.DataFrame([{
        "HomeTeam": HomeTeam,
        "AwayTeam": AwayTeam,
        "LocalGoals": LocalGoals,
        "AwayGoals": AwayGoals,
        "Final_result": Final_result,
        "HomeShots": HomeShots,
        "AwayShots": AwayShots,
        "HomeFauls": HomeFauls,
        "AwayFauls": AwayFauls,
        "HomeCorners": HomeCorners,
        "AwayCorners": AwayCorners,
        "HomeReds": HomeReds,
        "AwayReds": AwayReds,
    }])

    # Seleccionamos solo las features numéricas usadas en el modelo
    X_input = data[["LocalGoals","AwayGoals","HomeShots","AwayShots","HomeFauls",
                    "AwayFauls","HomeCorners","AwayCorners","HomeReds","AwayReds"]]

    # Escalamos
    X_scaled = scaler.transform(X_input)

    # Predicción
    pred_home_yellow = yellow_model.predict(X_scaled)[0]

    return pred_home_yellow


#Ejemplo de resolucion

prediccion_tarjetas = predecir_tarjetas(
    HomeTeam = "Arsenal",
    AwayTeam = "Chelsea",
    LocalGoals = 2,
    AwayGoals = 1,
    Final_result = 'H',
    HomeShots = 10,
    AwayShots = 7,
    HomeFauls=8,
    AwayFauls=10,
    HomeCorners=6,
    AwayCorners=4,
    HomeReds=0,
    AwayReds=0
)
print("Predecir tarjetas amarillas en un partido como este: ", prediccion_tarjetas)


**8) Prediccion interactiva--> nueva columnas (DifferentialPossesion)**

In [ ]:
#prediccion interactiva--> nueva columana = DifferentialPossesion

df['HomePossession'] = (df['HomeShots'] + df['HomeCorners']+ df['AwayFauls']+ df['AwayYellows']+ df['AwayReds'])
df['AwayPossession'] = (df['AwayShots'] + df['AwayCorners']+ df['HomeFauls']+ df['HomeYellows']+ df['HomeReds'])

#calcular possesion en cada partido, teniendo en cuenta el equipo local y visitante.

total = df["HomePossession"] + df["AwayPossession"]
df["HomePossession"] = 100 * df["HomePossession"] / total
df["AwayPossession"] = 100 * df["AwayPossession"] / total

df.head(5)



**Verificar ajuste mediante muchos bosques**

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
#random forest

X= df[['LocalGoals','AwayGoals','HomeShots','AwayShots','HomeFauls', 'AwayFauls','HomeCorners','AwayCorners']]
y= df['HomePossession'] - df['AwayPossession']

#calcular modelos de entrenamiento (train) y de evaluadoras (test)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=42)

random_f = RandomForestRegressor(n_estimators =1000, random_state = 42 )
random_f.fit(X_train,  y_train)

random_f_score= random_f.score(X_test, y_test)
print(f"Score del conjunto de prueb (r2): ", {random_f_score})

#Predice bien.

**Uso de widgets(eleccion de datos) para predecir la posesion**

In [ ]:
import ipywidgets as widgets
from IPython.display import display

#crear widgets para ingresar valores a las variables

LocalGoals_widget = widgets.FloatText(description = 'LocalGoals')
AwayGoals_widget = widgets.FloatText(description = 'AwayGoals')
HomeShots_widget= widgets.FloatText(description = 'HomeShots')
AwayShots_widget= widgets.FloatText(description = 'AwayShots')
HomeFauls_widget= widgets.FloatText(description = 'HomeFauls')
AwayFauls_widget= widgets.FloatText(description = 'AwayFauls')
HomeCorners_widget= widgets.FloatText(description = 'HomeCorners')
AwayCorners_widgets = widgets.FloatText(description = 'AwayCorners')

button = widgets.Button(description = 'Predecir') #funcion de entrada y salida del boton
output = widgets.Output()

#funcion que realiza la prediccion al pulsar el botton
def press_button(b):
  with output:
    output.clear_output() #limpiar salida anterior
    LocalGoals= LocalGoals_widget.value
    AwayGoals= AwayGoals_widget.value
    HomeShots= HomeShots_widget.value
    AwayShots= AwayShots_widget.value
    HomeFauls= HomeFauls_widget.value
    AwayFauls= AwayFauls_widget.value
    HomeCorners= HomeCorners_widget.value
    AwayCorners= AwayCorners_widgets.value

    #crear dataframe con valores de entrada y nombres de columnas

    input_data = pd.DataFrame([[LocalGoals, AwayGoals, HomeShots, AwayShots, HomeFauls, AwayFauls, HomeCorners, AwayCorners]],
                              columns = ['LocalGoals', 'AwayGoals', 'HomeShots', 'AwayShots','HomeFauls',
                                         'AwayFauls','HomeCorners','AwayCorners'] )

    #realizar la prediccion utlizando el modelo ya entrando en la celda anterior {randomForest}
    prediction = random_f.predict(input_data)[0]
    #Mostrar prediccion
    print(f"la prediccion para la diferencia de posesion (teniendo en cuenta las caracteristicas de entrada) es: {prediction:.2f} " )

    if (prediction > 0):
      print("El equipo local es el dominante posicionalmente.")
    else:
      print("El equipo visitante es el dominante posicionalmente.")

#conectar el boton a la funcion de prediccion que esta arriba
button.on_click(press_button)

#Mostrar widgets y el boton
display(LocalGoals_widget,AwayGoals_widget, HomeShots_widget, AwayShots_widget, HomeFauls_widget, AwayFauls_widget, HomeCorners_widget,
        AwayCorners_widgets, button, output)


si la prediccion es:>0, el equipo local es el dominante posisionalmente..

si la prediccion es:<0, el equipo visitante es el dominante posisionalemnte

**9) “Tabla de posiciones” predicha (demostración) (paul)**

Para mostrar una salida interpretable, convertimos predicciones de partidos en puntos (3/1/0) y agregamos por equipo. ⚠️ Limitación importante: si el modelo predice pocos empates, la suma total de puntos sube (porque los empates reparten 2 puntos en total vs 3 en victorias).

In [ ]:
from collections import defaultdict

def season_points_from_results(df_matches, result_col):
    """Agrega puntos por equipo a partir de resultados H/D/A."""
    pts = defaultdict(int)
    for r in df_matches.itertuples(index=False):
        res = getattr(r, result_col)
        pts[r.HomeTeam] += points_from_result(res, True)
        pts[r.AwayTeam] += points_from_result(res, False)
    return dict(pts)

test_with_pred = test[["HomeTeam","AwayTeam","y"]].copy()
test_with_pred["pred"] = pred

actual_pts = season_points_from_results(test_with_pred.rename(columns={"y":"Result"}), "Result")
pred_pts   = season_points_from_results(test_with_pred.rename(columns={"pred":"Result"}), "Result")

table = pd.DataFrame({
    "ActualPoints": pd.Series(actual_pts),
    "PredPoints":   pd.Series(pred_pts)
}).fillna(0).astype(int)

table["Diff"] = table["PredPoints"] - table["ActualPoints"]

table.sort_values("ActualPoints", ascending=False).head(10)

In [ ]:
plt.figure()
plt.scatter(table["ActualPoints"], table["PredPoints"])
plt.title("Puntos reales vs puntos predichos (2024–25)")
plt.xlabel("Puntos reales")
plt.ylabel("Puntos predichos")

# Línea horizontal - por ejemplo en el promedio de puntos predichos
mean_pred_points = table["PredPoints"].mean()
plt.axhline(y=mean_pred_points, color='r', linestyle='--', alpha=0.7,
            label=f'Media predicha: {mean_pred_points:.1f}')

# Etiquetamos top real para que sea legible
for team, row in table.iterrows():
    if row["ActualPoints"] >= table["ActualPoints"].quantile(0.90):
        plt.text(row["ActualPoints"], row["PredPoints"], team, fontsize=7)

plt.show()